## Foundational RL with Azure AI Fine-Tuning Sessions

This notebook demonstrates a synchronous, on-policy RL loop for tool calling with the published Python SDK. The public text-model default is **Qwen/Qwen3.8-27B**; this notebook selects its non-thinking renderer for the short tool-call budget.

### Before running

1. From the cookbook root, install the pinned PyPI SDK and notebook dependencies as described in [the installation guide](../../README.md#install): `python -m pip install -e '.[notebook]' --index-url https://pypi.org/simple --extra-index-url https://download.pytorch.org/whl/cpu`. Select that virtual environment as the notebook kernel.
2. Set `AZURE_AI_PROJECT_ENDPOINT` to your Foundry project URL, for example `https://<your-project>.services.ai.azure.com/api/projects/<project-name>`. The older `FOUNDRY_PROJECT_ENDPOINT` name is also accepted.
3. Set `AZURE_AI_API_KEY`, or authenticate with `az login`. See [authentication](../../docs/auth.md) and [supported models and regions](../../docs/supported_models.md).
4. The dataset is downloaded from Hugging Face. **Running the session and training cells uses remote compute and may incur charges.** The default settings are an example, not measured Qwen3.8 results.
5. Run the cleanup cell if you stop before the training loop. The loop also closes the session in `finally`. Shared notebook files should have saved outputs cleared to avoid publishing session IDs, data, or old model results.

The notebook uses these SDK operations:

| Operation | Purpose |
| --- | --- |
| `sample` | Generate completions from a tokenized prompt. |
| `forward_backward_async` | Accumulate gradients for rendered training data. |
| `optim_step_async` | Apply an optimizer update. |
| `save_weights_and_get_sampling_client_async` | Sync the current policy to a sampler before rollouts. |
| `save_weights_async` | Persist a training checkpoint. |
| `save_weights_for_sampler_async` | Persist a sampler checkpoint. |

At step $t$, the loop samples from the current policy $\pi_t$ and uses those rollouts to update it to $\pi_{t+1}$. Off-policy training would instead use rollouts from an older policy and requires separate importance-sampling and policy-staleness controls.

In [ ]:
from dataclasses import dataclass
import uuid

from interactive_training import model_info


@dataclass(frozen=True)
class Config:
    model_name: str = model_info.DEFAULT_MODEL_NAME
    seed: int = 42

    # Training: examples per run, prompts per update, completions per prompt.
    train_samples: int = 512
    batch_size: int = 8
    group_size: int = 8
    lora_rank: int = 16
    lora_alpha: float = 32

    learning_rate: float = 1e-5
    beta1: float = 0.9
    beta2: float = 0.95
    eps: float = 1e-8

    eval_samples: int = 50
    eval_every: int = 10
    save_every: int = 10

    # None selects a model-compatible non-thinking renderer where available.
    # For Qwen3.8 this is qwen3_8_disable_thinking, not qwen3_disable_thinking.
    renderer_name: str | None = None
    max_tokens: int = 512
    train_temperature: float = 1.1
    eval_temperature: float = 0.1
    top_p: float = 1.0
    top_k: int = -1
    train_sample_concurrency: int = 2
    eval_sample_concurrency: int = 4
    sample_timeout_sec: float = 300.0

    @property
    def train_steps(self) -> int:
        return (self.train_samples + self.batch_size - 1) // self.batch_size


CONFIG = Config()
CONFIG

In [ ]:
import sys
from pathlib import Path

# Support kernels started in the repo root, notebooks/, or this notebook's folder.
candidates = (
    Path.cwd() / "notebooks" / "foundational",
    Path.cwd() / "foundational",
    Path.cwd(),
)
NOTEBOOK_DIR = next((path for path in candidates if (path / "xlam_utils.py").is_file()), None)
if NOTEBOOK_DIR is None:
    raise RuntimeError("Start the kernel in the cookbook root, notebooks/, or notebooks/foundational/.")
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

In [ ]:
from datasets import load_dataset

from xlam_utils import xlam_expected_tool_calls, xlam_tools_to_specs


num_samples = CONFIG.eval_samples + CONFIG.train_samples
raw = load_dataset("Salesforce/xlam-function-calling-60k", split="train")
raw = raw.shuffle(seed=CONFIG.seed)

rows = []
for row in raw:
    row = dict(row)
    if row.get("query") and xlam_tools_to_specs(row) and xlam_expected_tool_calls(row):
        rows.append(row)
    if len(rows) >= num_samples:
        break

if len(rows) < num_samples:
    raise ValueError(f"Requested {num_samples} valid rows, but the dataset supplied only {len(rows)}.")
eval_rows = rows[: CONFIG.eval_samples]
train_rows = rows[CONFIG.eval_samples :]
print(f"Loaded {len(train_rows)} train rows and {len(eval_rows)} eval rows")

In [ ]:
import os
from urllib.parse import urlparse

from azure.ai.finetuningsessions.aio import FineTuningSessionClient
from azure.ai.finetuningsessions.models import LoRAConfig
from azure.core.credentials import AzureKeyCredential
from azure.identity.aio import DefaultAzureCredential

PROJECT_ENDPOINT = (
    os.environ.get("AZURE_AI_PROJECT_ENDPOINT")
    or os.environ.get("FOUNDRY_PROJECT_ENDPOINT")
    or ""
).strip()
endpoint_url = urlparse(PROJECT_ENDPOINT)
if endpoint_url.scheme != "https" or not endpoint_url.hostname or "<" in PROJECT_ENDPOINT:
    raise ValueError(
        "Set AZURE_AI_PROJECT_ENDPOINT to your Foundry project URL: "
        "https://<your-project>.services.ai.azure.com/api/projects/<project-name>. "
        "See docs/auth.md; do not put credentials in notebook source."
    )

api_key = os.environ.get("AZURE_AI_API_KEY")
credential = AzureKeyCredential(api_key) if api_key else DefaultAzureCredential()
client_options = {} if api_key else {"credential_scopes": ["https://ai.azure.com/.default"]}
client = FineTuningSessionClient(endpoint=PROJECT_ENDPOINT, credential=credential, **client_options)
session_id = None
session_closed = False
client_closed = False
credential_closed = bool(api_key)
resources_closed = False


async def close_training_session():
    """Retry unfinished cleanup without closing resources needed for remote retry."""
    global session_closed, client_closed, credential_closed, resources_closed
    if resources_closed:
        return

    # A failed or cancelled remote close leaves the client and credential usable.
    # Mark each step complete only after it succeeds, so reruns retry that step.
    if not session_closed:
        if session_id is not None:
            await client.close_session(session_id)
        session_closed = True
    if not client_closed:
        await client.close()
        client_closed = True
    if not credential_closed:
        await credential.close()
        credential_closed = True
    resources_closed = True


try:
    session_id = await client.create_session(
        base_model=CONFIG.model_name,
        lora_config=LoRAConfig(rank=CONFIG.lora_rank, alpha=CONFIG.lora_alpha, seed=CONFIG.seed),
        type="training",
        timeout_sec=900.0,
    )
except BaseException:
    await close_training_session()
    raise

session_id

In [ ]:
from interactive_training import model_info
from interactive_training.renderers import get_renderer
from interactive_training.rl.train_azure import AzureSDKTrainingClient
from interactive_training.tokenizer_utils import get_tokenizer

from sampling import sample_row

try:
    tokenizer = get_tokenizer(CONFIG.model_name)
    renderer_name = CONFIG.renderer_name or model_info.get_recommended_renderer_name(
        CONFIG.model_name, prefer_non_thinking=True
    )
    renderer = get_renderer(renderer_name, tokenizer)
    training_client = AzureSDKTrainingClient(client, session_id, tokenizer)
except BaseException:
    await close_training_session()
    raise

print("model_name:", CONFIG.model_name)
print("tokenizer:", tokenizer.__class__.__name__)
print("renderer:", renderer.__class__.__name__)

In [ ]:
import asyncio
import json
import statistics

from azure.ai.finetuningsessions.models import AdamParams

from rollout_metrics import compute_rollout_metrics
from training import datums_from_groups

adam = AdamParams(
    learning_rate=CONFIG.learning_rate,
    beta1=CONFIG.beta1,
    beta2=CONFIG.beta2,
    eps=CONFIG.eps,
)


async def train_step(step, rows, sampling_client):
    sem = asyncio.Semaphore(CONFIG.train_sample_concurrency)

    async def run_group(index, row):
        async with sem:
            return await sample_row(
                sampling_client,
                row,
                CONFIG,
                renderer,
                num_samples=CONFIG.group_size,
                temperature=CONFIG.train_temperature,
                seed=CONFIG.seed + step * 100_000 + index,
            )

    raw_groups = await asyncio.gather(*[run_group(i, row) for i, row in enumerate(rows)], return_exceptions=True)
    failures = [group for group in raw_groups if isinstance(group, Exception)]
    groups = [group for group in raw_groups if not isinstance(group, Exception)]
    if failures:
        print(f"step {step}: skipped {len(failures)} failed sample groups; first error: {failures[0]}")

    datums, rewards = datums_from_groups(groups)
    fb_task = await client.forward_backward_async(
        session_id,
        datums,
        loss_fn="importance_sampling",
        loss_fn_config=None,
    )
    fb_result = await fb_task

    opt_task = await client.optim_step_async(session_id, adam)
    opt_result = await opt_task

    metrics = {
        "step": step,
        "train/reward": statistics.mean(rewards),
        # Actual training examples can be fewer than successful rollouts if token/logprob data is invalid.
        "train/datums": len(datums),
        "train/failed_sample_groups": len(failures),
        "optim/lr": CONFIG.learning_rate,
    }
    metrics.update(compute_rollout_metrics(groups, CONFIG.max_tokens))
    # Same sampled-token entropy estimate as the recipes (not entropy_loss:sum).
    sampling_logprobs = [lp for group in groups for sample in group for lp in sample["logprobs"]]
    if sampling_logprobs:
        metrics["optim/entropy"] = -statistics.mean(sampling_logprobs)
    metrics.update(getattr(fb_result, "metrics", {}) or {})
    # Recipes retain backend optimizer names; the dashboard reads skyrl.ai/grad_norm.
    metrics.update(getattr(opt_result, "metrics", {}) or {})
    return metrics

In [ ]:
import json
from importlib import reload
from pathlib import Path

import evals
import rollout_metrics
from tqdm.auto import tqdm

compute_rollout_metrics = reload(rollout_metrics).compute_rollout_metrics
run_eval = reload(evals).run_eval

metrics_path = Path("data") / f"metrics_{session_id}.jsonl"
metrics_path.parent.mkdir(parents=True, exist_ok=True)
print(f"Metrics log: {metrics_path}")

progress = tqdm(range(1, CONFIG.train_steps + 1), desc="training", unit="step")


def checkpoint_name(name):
    return f"{name}-{uuid.uuid4().hex[:4]}"


workload_failed = False
try:
    for step in progress:
        metrics = {}
        sampling_client = await training_client.save_weights_and_get_sampling_client_async(
            checkpoint_name(f"sampling-{step}")
        )

        if step == 1 or step % CONFIG.eval_every == 0 or step == CONFIG.train_steps:
            eval_metrics, eval_misses = await run_eval(
                sampling_client, eval_rows, CONFIG, renderer, name=f"eval-step-{step}"
            )
            # Evaluation precedes this update; saved weights are one update newer.
            eval_metrics["eval/policy_step"] = step - 1
            metrics.update(eval_metrics)
            print(f"eval step {step}: eval/reward: {eval_metrics['eval/reward']:.4f}")

        start = (step - 1) * CONFIG.batch_size
        batch = train_rows[start : step * CONFIG.batch_size]
        metrics.update(await train_step(step, batch, sampling_client))
        with metrics_path.open("a", encoding="utf-8") as stream:
            stream.write(json.dumps(metrics) + "\n")

        if step == CONFIG.train_steps or step % CONFIG.save_every == 0:
            save_name = checkpoint_name(f"{step}")
            save_task = await client.save_weights_async(
                session_id, save_name, step_number=step, metrics=metrics
            )
            await save_task
            sampler_save_task = await client.save_weights_for_sampler_async(session_id, save_name)
            await sampler_save_task
            print(f"save step {step}: train/reward: {metrics['train/reward']:.4f}")
except BaseException:
    workload_failed = True
    raise
finally:
    progress.close()
    try:
        await close_training_session()
    except Exception:
        if not workload_failed:
            raise
        print("Session cleanup failed after a training error; inspect the session in Foundry.")

In [ ]:
# Safe to rerun after the training loop, or if you stopped before starting it.
# If cleanup raises, rerun this cell, not the session-creation cell.
# Unfinished steps are retried; the client and credential stay usable until remote close succeeds.
# If failures persist, retain session_id and unload the session in Foundry.
await close_training_session()
print("Training session and clients closed.")